<font color=#009afa size=50>DAT5001 - Visualisation of Data</font>

<font size=50>1 - Cleaning Satellite Dataset for Presentation</font>

|**Assessment Code:** |PRES1|
|---|---|
|**Author:**|Tom Rowan, ST20285213|
|**Course:**| DAT5001_S2_25|
|**Course Tutor:**|Dr. Imtiaz Khan|


## Initialise Notebook

In [184]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [185]:
# Standard Libraries
from datetime import datetime
import sys
import time

# Pandas and numpy
import pandas as pd
import numpy as np

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Check for Internet Access

In [186]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

In [187]:
# Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.

# Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online


_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.planet4589.org',
                  'https://datascience.daemonchild.com']

if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    sys.exit('Stopping notebook running here.')


[INTERNET] Internet Connectivity Detected. Using Internet resources.


## Load the Datasets from the Internet

Reference Website:
https://www.planet4589.org/space/gcat/index.html

Tab Separated Value Files: 

- SatCat: https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv
- Events: https://planet4589.org/space/gcat/tsv/cat/ecat.tsv
- Payload: https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv   
- Org Data: https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv
- Launch sites: https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv

Documentation: https://www.planet4589.org/space/gcat/pdf/doc.pdf

Reference: "McDowell, J, 2020: General Catalog of Artificial Space Objects, https://planet4589.org/space/gcat", First accessed 24/10/2025

> VIII.2.1.1 The Standard (S) Satellite Catalog
>
> Since the late 1950s the US government has maintained a catalog of satellites (often referred to as the SATCAT). Each
satellite is given a sequential number (1 was the rocket stage from the first Sputnik). These numbers are sometimes
referred to as SSN numbers.

In [188]:
# This will get the very latest dataset from John McDowells's page. This dataset is updated daily!
satcat_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv', sep='\t')
events_df = pd.read_csv('https://planet4589.org/space/gcat/tsv/cat/ecat.tsv', sep='\t')
pay_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv', sep='\t')
orgs_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv', sep='\t')
lsite_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv', sep='\t')

datasets = {'satcat_df': satcat_df, 
            'events_df': events_df, 
            'pay_df': pay_df, 
            'orgs_df': orgs_df,
            'lsite_df': lsite_df
            }

Save a local copy

In [189]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
timestamp_formatted = today.strftime("%Y%m%d")

# Create filenames, timestamped
filepath = '../../datasets/'

for k,v in datasets.items():
    filename = f'{filepath}{k}_{timestamp_formatted}.csv'
    v.to_csv(filename, index=False)


### Explore these datasets

In [190]:
for k,v in datasets.items():
    print ("-----")
    print ("Dataset: " + k)
    print ("")
    print (v.columns)
    print ("")
    print (v.shape)

-----
Dataset: satcat_df

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object')

(66650, 42)
-----
Dataset: events_df

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object

## Data Cleaning
Remove columns that are not useful for the report.

**Satcat Database**

In [191]:
# Remove columns we can't use
columns_to_keep = ['Piece', 'Type', 'Name', 'LDate', 'DDate', 'Status', 'Owner', 'State', 'Manufacturer', 'Mass', 'Length', 'Diameter', 'Span', 'Shape', 'Perigee', 'Apogee']
sdf = satcat_df[columns_to_keep]

# Rename Columns for easier reading
sdf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date', 'Owner':'Operator Code','Manufacturer':'Manufacturer Code'}, inplace=True)

The date fields are a little challenging. Some contain just dates (US format), some contain date and times, and some contain "-" or " -". These are fixed now because other calculations will rely on solid dates.

In [192]:
sdf['Launch Date'] = sdf['Launch Date'].replace("-", np.nan)
sdf['Decay Date'] = sdf['Decay Date'].replace("-", np.nan)
sdf['Decay Date'] = sdf['Decay Date'].replace(" -", np.nan)

# Strip trailing time values: keep only first 3 tokens (year, month, day)
sdf['Launch Date'] = (
    sdf['Launch Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3
    .str.join(" ")      # join back into a string
)

sdf['Decay Date'] = (
    sdf['Decay Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3# Replace "-" with NaN

    .str.join(" ")      # join back into a string
)

# There are some empty rows - such as where JCAT = S49051
sdf = sdf.dropna(subset=['Launch Date'])


# There are some dates that are questionable - remove the question marks
sdf['Decay Date'] = sdf['Decay Date'].str.replace('?', '', regex=False)

In [193]:
sdf.shape

(66586, 16)

**Events Database**

In [194]:
edf_columns_to_keep = ['Piece', 'LDate', 'DDate', 'Status', 'Perigee', 'Apogee']
edf = events_df[edf_columns_to_keep]

edf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date'}, inplace=True)

In [195]:
edf['Decay Date'] = edf['Decay Date'].replace("-", np.nan)
edf['Decay Date'] = edf['Decay Date'].replace(" -", np.nan)

# Strip trailing time values: keep only first 3 tokens (year, month, day)
edf['Launch Date'] = (
    edf['Launch Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3
    .str.join(" ")      # join back into a string
)

edf['Decay Date'] = (
    edf['Decay Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3# Replace "-" with NaN

    .str.join(" ")      # join back into a string
)

# There are some dates that are questionable - remove the question marks
edf['Decay Date'] = edf['Decay Date'].str.replace('?', '', regex=False)

**Organisations Database**

In [196]:
# Create the CombinedName column using owner_EName if available, otherwise owner_Name
orgs_df['CombinedName'] = orgs_df['EName'].fillna(orgs_df['Name'])

# Remove columns we can't use
orgs_columns_to_keep = ['#Code', 'ShortName', 'CombinedName', 'Longitude', 'Latitude', 'Parent']
odf = orgs_df[orgs_columns_to_keep]

# Rename Columns for easier reading
odf.rename(columns={'#Code':'Org Code', 'ShortName':'Short Name', 'CombinedName':'Long Name'}, inplace=True)

Convert Latitude and Longitude to City and State values

In [ ]:
# Do this very rarely! It takes three hours due to 2 second API rate limits.
do_lat_lon_lookup = False

In [198]:
# Define a function to extract data from Nominatim database
def get_location_info(lat, lon):
    time.sleep(1.5)
    lat_lon = f'{lat},{lon}'
    
    try:
        location = geolocator.reverse(lat_lon, language="english")
        if location and location.raw.get('address'):
            address = location.raw['address']
            city = address.get('city', address.get('town', address.get('village', '')))
            state = address.get('state', '')
            country = address.get('country', '')
            countrycode = address.get('country_code', '')

            return f"{city}|{state}|{country}|{countrycode}"

    except GeocoderTimedOut:
        print("Error: Geocoding timed out. Please try again later.")
    except GeocoderServiceError as e:
        print(f"Service error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    
if do_lat_lon_lookup and online:
    print_output ('[CITY/STATE/COUNTRY DATA]','This is going to take three hours.',colour='green', delim='')
    
    # Set up the geocoder
    geolocator = Nominatim(user_agent="Cardiff_Met_Project_Tom_Rowan")

    # Apply the function using lambda
    odf['GeoInfo'] = odf.apply(lambda row: get_location_info(row['Latitude'], row['Longitude']), axis=1)

    odf[['City', 'State', 'Country', 'Country Code']] = odf['GeoInfo'].str.split('|', expand=True)
    odf.drop(columns=['GeoInfo'])
    odf.to_csv('../../datasets/odf-complete-version.csv', index=False)
else:
    print_output ('[CITY/STATE/COUNTRY DATA]','Loading from saved version',colour='green', delim='')
    odf = pd.read_csv('../../datasets/odf-complete-version.csv')

[CITY/STATE/COUNTRY DATA] This is going to take three hours.
An unexpected error occurred: Must be a coordinate pair or Point
An unexpected error occurred: Must be a coordinate pair or Point
An unexpected error occurred: Must be a coordinate pair or Point
An unexpected error occurred: Must be a coordinate pair or Point


In [199]:
odf.sample(20)

,Org Code,Short Name,Long Name,Longitude,Latitude,Parent,GeoInfo,City,State,Country,Country Code
3392,TELMX,Telmex,Tele Mexico,-99.1300,19.4300,-,Ciudad de México|Ciudad de México|México|mx,Ciudad de México,Ciudad de México,México,mx
2250,MMSD,MM/San Diego,-,-117.1400,32.8300,MM,San Diego|California|United States of America|us,San Diego,California,United States of America,us
3204,SPABD,SpaceBD,-,139.7500,35.6900,-,千代田区||日本|jp,千代田区,,日本,jp
3235,SPIRE,Spire,-,-122.4200,37.7800,-,San Francisco|California|United States of Amer...,San Francisco,California,United States of America,us
1086,DBP,Bundespost,DBP Telecom,8.6500,49.8700,-,Darmstadt|Hessen|Deutschland|de,Darmstadt,Hessen,Deutschland,de
2016,LIGACA,Ligado,-,-75.6100,45.4400,LIGA,Ottawa|Ontario|Canada|ca,Ottawa,Ontario,Canada,ca
1747,INSYS,Insys Ltd,-,-0.4900,52.0300,-,Ampthill|England|United Kingdom|gb,Ampthill,England,United Kingdom,gb
1092,DEHAVS,de Havilland,-,-0.2000,51.9000,-,Stevenage|England|United Kingdom|gb,Stevenage,England,United Kingdom,gb
738,BIT,BIT,Beijing Inst. of Technology,116.3100,39.9600,-,海淀区|北京市|中国|cn,海淀区,北京市,中国,cn
3289,SSTUS,SST-US,-,-104.9900,39.6500,SSTL,Englewood|Colorado|United States of America|us,Englewood,Colorado,United States of America,us


**Payload Database**

In [200]:
payload_columns_to_keep = ['Piece', 'Program', 'Class', 'Category']
pdf = pay_df[payload_columns_to_keep]

# Rename 'Piece' to 'CosparID'
pdf.rename(columns={'Piece':'CosparID'}, inplace=True)

## Combine the Datasets

In [201]:
# Rename odf columns for Operator merge
owner_odf = odf.rename(columns=lambda x: f'Operator {x}' if x != 'Org Code' else 'Operator Code')
sdf = sdf.merge(owner_odf, on='Operator Code', how='left')

# Repeat for Manufacturer merge
manufacturer_odf = odf.rename(columns=lambda x: f'Manufacturer {x}' if x != 'Org Code' else 'Manufacturer Code')
sdf = sdf.merge(manufacturer_odf, on='Manufacturer Code', how='left')

# Merge payload data
sdf = sdf.merge(pdf, on='CosparID', how='left')

In [202]:
sdf.columns

Index(['CosparID', 'Type', 'Name', 'Launch Date', 'Decay Date', 'Status',
       'Operator Code', 'State', 'Manufacturer Code', 'Mass', 'Length',
       'Diameter', 'Span', 'Shape', 'Perigee', 'Apogee', 'Operator Short Name',
       'Operator Long Name', 'Operator Longitude', 'Operator Latitude',
       'Operator Parent', 'Operator GeoInfo', 'Operator City',
       'Operator State', 'Operator Country', 'Operator Country Code',
       'Manufacturer Short Name', 'Manufacturer Long Name',
       'Manufacturer Longitude', 'Manufacturer Latitude',
       'Manufacturer Parent', 'Manufacturer GeoInfo', 'Manufacturer City',
       'Manufacturer State', 'Manufacturer Country',
       'Manufacturer Country Code', 'Program', 'Class', 'Category'],
      dtype='object')

### Using Events to Update the Dataset

In [203]:
sdf['Decay Date'].isna().value_counts()

Decay Date
False    35266
True     31320
Name: count, dtype: int64

The events database contains updates to the Satcat data, so these need to be applied like a database log to get the updated values.

For example, this ID shows as being docked.

In [204]:
sdf[sdf['CosparID'] == '1988-038A']['Status']

19116    DK
Name: Status, dtype: object

However, later on in the events database, it undocked (UDK) and then finally deorbited (D):

In [205]:
edf[edf['CosparID'] == '1988-038A']['Status']

767    UDK
770      D
Name: Status, dtype: object

In [206]:
# Ensure types are comparible
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], errors='coerce')

edf['Launch Date'] = pd.to_datetime(edf['Launch Date'], errors='coerce')
edf['Decay Date'] = pd.to_datetime(edf['Decay Date'], errors='coerce')

# Dedupe edf to the latest update per CosparID by index, therefore keeping the last update.
edf_latest = (
    edf.sort_index()                # ensure index order is respected
       .drop_duplicates('CosparID', keep='last')
)

# Merge and update only specified columns
merged = sdf.merge(edf_latest, on='CosparID', how='left', suffixes=('', '_update'))

# Columns to update from edf_latest
cols_to_update = ['Launch Date', 'Decay Date', 'Status', 'Perigee', 'Apogee']

for col in cols_to_update:
    update_col = f"{col}_update"
    if update_col in merged.columns:
        merged[col] = merged[update_col].combine_first(merged[col])

# Clean up _update columns
sdf = merged.drop(columns=[c for c in merged.columns if c.endswith('_update')])



Check with the example from above:

In [207]:
sdf[sdf['CosparID'] == '1988-038A']['Status']

19116    D
Name: Status, dtype: object

# Feature Engineering

### Category, Class and Type Fields

In [208]:
# These mappings were created from the documentation:

class_json = {
  "A": "Non-commercial",        # Academic / Amateur / Non-Profit
  "B": "Commercial",
  "C": "Government (Civil)",
  "D": "Government (Defence)",  # and Military and Intelligence
  "U": "Unknown"
}

type_json = {
  "P": "Payload",
  "C": "Component",
  "R": "Rocket Body",
  "D": "Debris",
  "S": "Suborbital payload",
  "X": "Deleted Object",
  "Z": "Spurious Entry"
}

category_json = {
  "AST": "Astronomy",
  "BIO": "Biology",
  "CAL": "Calibration",
  "COM": "Communications",
  "EDU": "Education",
  "EOSCI": "Earth observing science",
  "EW": "Missile early warning",
  "GEOD": "Geodesy",
  "IMG": "Imaging (optical)",
  "IMG-R": "Imaging (radar)",
  "INF": "Infrastructure",
  "MET": "Meteorology (imaging)",
  "MET-RO": "Meteorology (radio occultation)",
  "MGRAV": "Microgravity experiments",
  "MISC": "Miscellaneous",
  "NAV": "Navigation",
  "PLAN": "Deep space related mission",
  "RB": "Rocket body",
  "RV": "Reentry vehicle",
  "SCI": "Scientific studies",
  "SIG": "Signals intelligence",
  "SS": "Spaceship",
  "TARG": "Target",
  "TECH": "Technology",
  "WEAPON": "Weapon",
  "UNK": "Unknown"
}

status_json = {
    "O": "In orbit",
    "AO": "Attached in orbit",
    "AO IN": "Attached inside",
    "UDK": "Undocked",
    "REL": "Released",
    "DEP": "Deployed",
    "TO": "Takeoff Launch",
    "TOA": "Attached at takeoff",
    "OX": "In orbit",
    "E": "Exploded",
    "N": "Renamed",
    "NA": "Renamed attached",
    "LEASE": "Lease",
    "S": "Targeted suborbital entry/impact",
    "R": "Reentered",
    "D": "Deorbited",
    "L": "Landed",
    "LF": "Failed landing",
    "AS": "Attached Suborbital reentered",
    "AR": "Reentered attached",
    "AR IN": "Reentered inside",
    "AL": "Landed attached",
    "AL IN": "Landed inside",
    "TX": "Transmission ended",
    "F": "Failed",
    "AF": "Failed attached",
    "DK": "Docked",
    "GRP": "Grappled",
    "ATT": "Attached",
    "TFR": "Transfer Inside",
    "TFR E": "Transfer external Inside",
    "TFR IN": "Transfer in"
}

In [209]:
# Split the Category column at the first '/'
split_cols = sdf['Category'].str.split('/', n=1, expand=True)

# Remove trailing spurious characters from each part using a regex
sdf['Category'] = sdf['Category'].fillna('UNK')
sdf['Primary Category'] = split_cols[0].str.strip().str.replace(r'[*?]+$', '', regex=True) if split_cols.shape[0] > 1 else 'UNK'
sdf['Secondary Category'] = split_cols[1].str.strip().str.replace(r'[*?]+$', '', regex=True) if split_cols.shape[1] > 1 else 'UNK'

# Type field - Remove Sub-Types
sdf['Type'] = sdf['Type'].str[0]

# Remove X and Z objects ("Spurious and unknown")
sdf = sdf[~sdf['Type'].isin(['X', 'Z'])]

# Set Class to Unknown if not labelled
sdf['Class'] = sdf['Class'].fillna('U')

# Map the descriptions for better data visualisation labelling
sdf['Class Desc'] = sdf['Class'].map(class_json)
sdf['Type Desc'] = sdf['Type'].map(type_json)
sdf['Primary Category Desc'] = sdf['Primary Category'].map(category_json)
sdf['Secondary Category Desc'] = sdf['Secondary Category'].map(category_json)
sdf['Status Desc'] = sdf['Status'].map(status_json)

Delete coded columns and then swap round so that "Type Desc" becomes just "Type"

In [210]:
sdf.drop(columns=['Type', 'Class', 'Category', 'Primary Category', 'Secondary Category', 'Status'], inplace=True, axis=1)
sdf.rename(columns={'Type Desc':'Type'}, inplace=True)

# Rename columns ending with " Desc"
sdf = sdf.rename(
    columns=lambda c: c[:-5] if c.endswith(" Desc") else c
)

In [211]:
sdf['Type'].value_counts()

Type
Debris         27825
Payload        24635
Component       7905
Rocket Body     6195
Name: count, dtype: int64

In [212]:
sdf['Class'].value_counts()

Class
Unknown                 41837
Commercial              15246
Government (Defence)     5420
Government (Civil)       3154
Non-commercial            883
Name: count, dtype: int64

In [213]:
sdf['Primary Category'].value_counts()

Primary Category
Communications                     15097
Imaging (optical)                   2998
Technology                          1570
Calibration                          811
Spaceship                            765
Signals intelligence                 701
Navigation                           623
Scientific studies                   521
Meteorology (imaging)                303
Imaging (radar)                      268
Deep space related mission           257
Astronomy                            218
Missile early warning                213
Earth observing science              117
Geodesy                               65
Meteorology (radio occultation)       63
Weapon                                37
Microgravity experiments              31
Biology                               30
Target                                21
Reentry vehicle                       10
Education                              4
Name: count, dtype: int64

In [214]:
sdf['Status'].value_counts()

Status
Reentered             32218
In orbit              31634
Landed                 1326
Deorbited               401
Reentered attached      300
Attached in orbit        86
Landed attached          24
Lease                    14
Exploded                 12
Landed inside             3
Transfer Inside           3
Released                  2
Reentered inside          1
Renamed                   1
Name: count, dtype: int64

### Is the object still in orbit?

Work out whether the object is still in orbit by combining specific status wording and whether there is a Decayed Date recorded or not.

In [215]:
# Define category lists
not_in_orbit_categories = ['Reentered', 'Landed', 'Deorbited', 'Reentered attached', 'Reentered inside', 'Landed attached', 'Landed inside']
in_orbit_categories = ['In Orbit', 'Attached in orbit', 'Docked']

# If not in any of these categories, but has a decay date
sdf['In Orbit'] = (
    ( ~sdf['Status'].isin(not_in_orbit_categories) & ~sdf['Status'].isin(in_orbit_categories) & sdf['Decay Date'].notna() )
)

# Create the new column 'In Orbit'
sdf['In Orbit'] = (
    sdf['Status'].isin(in_orbit_categories) |       # Found in "In Orbit category"
    (~sdf['Status'].isin(not_in_orbit_categories))  # Not found in "No In Orbit category"
)



Returning to the example from before:

In [216]:
sdf[sdf['CosparID'] == '1988-038A']['In Orbit']

19116    False
Name: In Orbit, dtype: bool

In [217]:
sdf.sample(20)[['CosparID','Type', 'Status', 'Launch Date', 'Decay Date','In Orbit']]

,CosparID,Type,Status,Launch Date,Decay Date,In Orbit
51222,1982-092BBE,Debris,Reentered,1982-09-16,2022-09-21,False
28003,1988-085BA,Debris,In orbit,1988-09-16,NaT,True
39133,2013-015E,Payload,Reentered,2013-04-19,2023-09-06,False
20529,1990-024B,Rocket Body,Reentered,1990-03-22,1990-03-23,False
36031,2009-058A,Payload,In orbit,2009-10-29,2009-11-09,True
51934,2022-025L,Payload,In orbit,2022-03-09,NaT,True
23184,1994-043A,Payload,In orbit,1994-07-21,NaT,True
29768,1999-025BK,Debris,In orbit,1999-05-10,NaT,True
51610,2022-012T,Payload,In orbit,2022-02-10,NaT,True
18820,1988-005B,Rocket Body,In orbit,1988-01-30,NaT,True


In [218]:
sdf['In Orbit'].value_counts().reset_index()

,In Orbit,count
0,False,34273
1,True,32287


In [219]:
# Check the results looks sane
filtered_df = sdf[(sdf['Type'] == 'Payload')]
                  
pivot_table = filtered_df.pivot_table(
    index='Status',
    columns='In Orbit',
    aggfunc='size',
    fill_value=0
)

# Display the result
print(pivot_table)

In Orbit            False  True 
Status                          
Attached in orbit       0     85
Deorbited             362      0
Exploded                0      8
In orbit                0  16722
Landed               1325      0
Landed attached        24      0
Landed inside           3      0
Lease                   0     14
Reentered            5449      0
Reentered attached    298      0
Released                0      1
Renamed                 0      1
Transfer Inside         0      1


### Generate Additional Date Values

The dates that are available are:

- Launch Date (always available)
- Decay Date (may be NULL or '?')

These values are converted from strings in the base dataset into python datetime objects.
This allows for comparisons and for additional values ot be inferred from them.

The following date columns were created:

- Launch Year
- Launch Month (Numerical)
- Launch Month (Text)
- Launch Decade (1950's: is 1950-1959.)
- Decay Year
- Decay Month (Numerical)
- Decay Month (Text)
- Decay Decade 

A temporary value for 'Effective Decay Date' is either copied from 'Decay Date' or set to 'Today' if the object is "In Orbit".
This allows for lifetime values in days, months and years to be calculated.

Finally, the datetime objects are reverted to strings (in UK format).

In [220]:
# Modify date columns
# Convert to datetime format
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], format='%Y %b %d', errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], format='%Y %b %d', errors='coerce')

# Create a Year of Launch and Decay
sdf['Launch Year'] = sdf['Launch Date'].dt.year.astype('Int64')
sdf['Decay Year'] = sdf['Decay Date'].dt.year.astype('Int64')
sdf['Launch Month No'] = sdf['Launch Date'].dt.month.astype('Int64')
sdf['Decay Month No'] = sdf['Decay Date'].dt.month.astype('Int64')
sdf['Launch Month'] = sdf['Launch Date'].dt.strftime('%B')
sdf['Decay Month'] = sdf['Decay Date'].dt.strftime('%B')
sdf['Launch Decade'] = ((sdf['Launch Year'] // 10) * 10)
sdf['Decay Decade'] = ((sdf['Decay Year'] // 10) * 10)

# Replace null Decay Date with today's date
#sdf['Effective Decay Date'] = sdf['Decay Date'].fillna(today)
# If Status == "In Orbit", set to today, else use Decay Date
sdf['Effective Decay Date'] = np.where(
    sdf['In Orbit'] == True,
    today,
    sdf['Decay Date']
)

# Convert to datetime 
sdf['Effective Decay Date'] = pd.to_datetime(sdf['Effective Decay Date'], errors='coerce')

# Calculate lifetime
delta = sdf['Effective Decay Date'] - sdf['Launch Date']
sdf['Lifespan Days'] = delta.dt.days
sdf['Lifespan Months'] = (delta.dt.days / 30.44).round(1)  # Approximate month length
sdf['Lifespan Years'] = (delta.dt.days / 365.25).round(2)  # Account for leap years

# Format as dd/mm/yyyy (because we are British)
sdf['Launch Date'] = sdf['Launch Date'].dt.strftime('%d/%m/%Y')
sdf['Decay Date'] = sdf['Decay Date'].dt.strftime('%d/%m/%Y')

# Drop Effective Decay Date column
sdf = sdf.drop('Effective Decay Date', axis=1)

In [221]:
sdf.sample(20)[['CosparID','Type', 'Status', 'Launch Date', 'Decay Date','Lifespan Days','In Orbit']]

,CosparID,Type,Status,Launch Date,Decay Date,Lifespan Days,In Orbit
5296,1971-056A,Payload,Deorbited,15/06/1971,06/08/1971,52.0,False
62112,2024-225C,Payload,In orbit,30/11/2024,NaN,363.0,True
25168,1998-010A,Payload,Reentered,18/02/1998,05/11/2018,7565.0,False
61411,2024-140ZW,Debris,Reentered,06/08/2024,01/10/2025,421.0,False
44936,2020-001Z,Payload,Reentered,07/01/2020,11/08/2025,2043.0,False
31837,1999-025CJC,Debris,In orbit,10/05/1999,NaN,9699.0,True
65470,2025-200J,Payload,In orbit,06/09/2025,NaN,83.0,True
29873,1999-025FU,Debris,In orbit,10/05/1999,NaN,9699.0,True
24317,1996-049D,Component,Reentered,21/08/1996,21/12/2001,1948.0,False
21726,1991-067B,Rocket Body,In orbit,26/09/1991,NaN,12482.0,True


### Map Country Code to Country Name

The country codes ('State') supplied are two figure codes. 
These are first converted to three figure codes, then a Country name derived from this.

In [222]:
# Convert alpha-2 to alpha-3
def alpha2_to_alpha3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

# Apply conversion to your State column
sdf['Country Code'] = sdf['State'].apply(alpha2_to_alpha3)
sdf['Country'] = sdf['Country Code'].apply(lambda x: pycountry.countries.get(alpha_3=x).name if x else None)

# Drop State Column
sdf.drop(columns='State', axis=1, inplace=True)

In [223]:
# Create a combined "Program Root" field
#sdf["Program Root"] = sdf["Program"].str.split().str[0]
sdf["Program Root"] = sdf["Program"].str.extract(r"^([A-Za-z0-9]+)")

Tidy Apogee and Perigee columns

In [224]:
# Convert to numeric, non-numeric → NaN
sdf['Apogee'] = pd.to_numeric(sdf['Apogee'], errors='coerce')
sdf['Perigee'] = pd.to_numeric(sdf['Perigee'], errors='coerce')

# Flip negatives to positive
sdf['Apogee'] = sdf['Apogee'].abs()
sdf['Perigee'] = sdf['Perigee'].abs()

In [225]:
# Reorder columns alphabetically
sdf = sdf.reindex(sorted(sdf.columns), axis=1)

In [226]:
sdf.columns

Index(['Apogee', 'Class', 'CosparID', 'Country', 'Country Code', 'Decay Date',
       'Decay Decade', 'Decay Month', 'Decay Month No', 'Decay Year',
       'Diameter', 'In Orbit', 'Launch Date', 'Launch Decade', 'Launch Month',
       'Launch Month No', 'Launch Year', 'Length', 'Lifespan Days',
       'Lifespan Months', 'Lifespan Years', 'Manufacturer City',
       'Manufacturer Code', 'Manufacturer Country',
       'Manufacturer Country Code', 'Manufacturer GeoInfo',
       'Manufacturer Latitude', 'Manufacturer Long Name',
       'Manufacturer Longitude', 'Manufacturer Parent',
       'Manufacturer Short Name', 'Manufacturer State', 'Mass', 'Name',
       'Operator City', 'Operator Code', 'Operator Country',
       'Operator Country Code', 'Operator GeoInfo', 'Operator Latitude',
       'Operator Long Name', 'Operator Longitude', 'Operator Parent',
       'Operator Short Name', 'Operator State', 'Perigee', 'Primary Category',
       'Program', 'Program Root', 'Secondary Category', 'S

In [227]:
sdf.shape

(66560, 54)

### Write out the cleaned dataset for further use



In [228]:
# Create filenames, one timestamped, one 'latest'
filename = f'{filepath}space_objects_{timestamp_formatted}.csv'
latestname = f'{filepath}space_objects_latest.csv'

# Write out CSV files
sdf.to_csv(filename, index=False)
sdf.to_csv(latestname, index=False)